# Channel data

> The data to train a power control that predicts the channel: ray-traced layouts of a stable-marl environment, the agents' trajectories in them, and per link and step, what the transmitter knows and the gain it gets.

In [ ]:
#| default_exp comm.channel_data

In [ ]:
#| hide
from nbdev.showdoc import *

The transmitter of each directed link must set its power before it knows the channel. A learned predictor (see
`comm.power_control`) estimates the gain from what the transmitter does know: the link's past gains, whether the
agents moved, where the receiver was a step ago, and the layout's walls. This module produces its training data:

1. **Trajectories**: `collect_trajectories` rolls agents out in the layout of each reset seed (stable-marl's
   `MixedPolicy`) and keeps every agent's cell and heading at every step. They are kept raw, so the features can
   change without collecting again.
2. **Channels**, ray-traced (Sionna RT): only the pairs of cells the trajectories visit (`trace_visited_pairs`, into
   a `PairCache` per layout), or every pair of free cells (`trace_layouts`). Agents visit a few percent of a layout's
   pairs, so the first is far cheaper.
3. **Samples**, built from the two: `link_samples` and `build_dataset` give, per directed link and step, the features
   and the gain.

In experiments, `CachedRayTracedFading` gives comm-core the channel from a layout's `PairCache`, and traces the pairs
it has not seen yet when they first occur.

The environment comes from a config with a `_target_` (Hydra style), e.g. `configs/comm/env/findgoal.yaml`. The
layout of a stable-marl MultiGrid environment is decided by its reset seed, so a layout is named by its seed.

In [ ]:
#| export
from __future__ import annotations

import importlib
import json
import math
from pathlib import Path

import numpy as np

## The environment

In [ ]:
#| export
DIRS = [(1, 0), (0, 1), (-1, 0), (0, -1)]       # MultiGrid's headings: right, down, left, up

def _plain(cfg) -> dict:
    "A config as a plain dict (an OmegaConf config resolved)."
    try:
        from omegaconf import DictConfig, OmegaConf
        if isinstance(cfg, DictConfig):
            return OmegaConf.to_container(cfg, resolve=True)
    except ImportError:
        pass
    return dict(cfg)

def make_env(env_cfg):
    "A stable-marl environment from its config: a mapping with the class as `_target_` (as Hydra's) and its arguments."
    kwargs = _plain(env_cfg)
    target = kwargs.pop('_target_')
    kwargs.pop('_convert_', None)
    module, name = target.rsplit('.', 1)
    return getattr(importlib.import_module(module), name)(**kwargs)

In [ ]:
#| export
def free_cells(env) -> list[tuple[int, int]]:
    "The cells an agent can stand on (empty, or that can be overlapped, e.g. the goal), column by column."
    g = env.unwrapped.grid
    return [(x, y) for x in range(g.width) for y in range(g.height) if (o := g.get(x, y)) is None or o.can_overlap()]

def reset_in_layout(env, layout_seed: int, rng: np.random.Generator):
    """Reset `env` to the layout of `layout_seed`, then move the goal and the agents to random free cells (random
    headings): many episodes in one layout. (A reset seed fixes the goal and the starts as well as the layout.)"""
    from stable_marl.envs.multigrid.core.world_object import Goal
    env.reset(seed=int(layout_seed))
    u = env.unwrapped
    goal = getattr(u, 'goal_pos', None)
    if goal is not None:
        u.grid.set(int(goal[0]), int(goal[1]), None)
    cells = free_cells(env)
    chosen = rng.choice(len(cells), len(u.agents) + (goal is not None), replace=False)
    if goal is not None:
        u.goal_pos = np.asarray(cells[chosen[-1]])
        u.grid.set(*cells[chosen[-1]], Goal())
    for agent, k in zip(u.agents, chosen):
        agent.state.pos = cells[k]
        agent.state.dir = int(rng.integers(4))
    return env

## Layouts

`trace_layouts` ray-traces the layout of each seed into `<out_dir>/layout_<seed>.npz`: `H[rx, tx]` (complex, summed
over the paths), the free `cells` in H's order, the `walls` grid (`walls[x, y]` = 1 where an agent cannot stand), and
what produced them (`source`). It needs Sionna RT and comm-core. Tracing is the expensive step: its time grows with
the square of the number of free cells (a 9 × 9 FindGoal grid, 39 free cells: 2–5 minutes on a CPU; 15 × 15 with 6
obstacles, about 140: roughly 13 times longer), much faster on a GPU. Layouts already traced are kept, so a long
run can be split over jobs (`seeds[i::n]`).

In [ ]:
#| export
def _scene_config(scene: dict | None):
    "stable-marl's `SceneConfig`, with `scene`'s fields replaced (dict fields, e.g. `materials`, updated)."
    from stable_marl.envs.multigrid.sionna_export import SceneConfig
    config = SceneConfig()
    for key, value in (scene or {}).items():
        if isinstance(getattr(config, key), dict):
            getattr(config, key).update(value)
        else:
            setattr(config, key, value)
    return config

def trace_layouts(
    env_cfg, # The environment's config (`make_env`)
    seeds, # Reset seeds: one layout each
    out_dir, # Where `layout_<seed>.npz` go
    frequency: float = 2.4e9, # Carrier frequency (Hz)
    max_depth: int = 5, # Reflections per path
    scene: dict | None = None, # `SceneConfig` fields to change, e.g. {'materials': {'inner_wall': 'brick'}}
    overwrite: bool = False, # Trace again the layouts already in `out_dir`
    progress: bool = True, # Print each layout's tracing time
    **solver_kwargs, # Other options of Sionna RT's PathSolver, e.g. samples_per_src
) -> list[Path]:
    "Ray-trace the layout of each reset seed into `<out_dir>/layout_<seed>.npz`."
    import time
    from stable_marl.envs.multigrid.sionna_export import trace_channel_table
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for seed in seeds:
        path = out_dir / f'layout_{seed}.npz'
        paths.append(path)
        if path.exists() and not overwrite:
            continue
        env = make_env(env_cfg)
        env.reset(seed=int(seed))
        start = time.time()
        table = trace_channel_table(env, _scene_config(scene), frequency, max_depth, **solver_kwargs)
        source = dict(env=_plain(env_cfg), seed=int(seed), frequency=frequency, max_depth=max_depth, scene=scene or {},
                      solver=solver_kwargs)
        np.savez_compressed(path, H=table['H'].astype(np.complex64), cells=table['cells'].astype(np.int16),
                            walls=table['walls'].astype(np.uint8), source=json.dumps(source, default=str))
        if progress:
            print(f'layout {seed}: {len(table["cells"])} cells, traced in {time.time() - start:.0f} s', flush=True)
    return paths

In [ ]:
#| export
def wall_geometry(walls: np.ndarray, a, b) -> tuple[float, int]:
    "Distance (cells) between cells `a` and `b`, and the number of wall cells the straight line between their centres crosses."
    n = max(int(abs(a[0] - b[0]) + abs(a[1] - b[1])) * 4, 1)
    xs, ys = np.linspace(a[0], b[0], n + 1), np.linspace(a[1], b[1], n + 1)
    crossed = {(int(round(x)), int(round(y))) for x, y in zip(xs, ys)} - {tuple(a), tuple(b)}
    return math.hypot(a[0] - b[0], a[1] - b[1]), int(sum(walls[c] for c in crossed))

def _pair(a, b) -> tuple:
    "An unordered pair of cells: a sorted tuple of two (x, y) tuples."
    a, b = tuple(int(v) for v in a), tuple(int(v) for v in b)
    return (a, b) if a <= b else (b, a)

class ChannelTable:
    "A traced layout: the gain between two of its free cells, and its walls. From a full table, or a `PairCache`'s pairs."

    def __init__(
        self,
        path, # A `layout_<seed>.npz` of `trace_layouts`, or a `pairs_<seed>.npz` of a `PairCache`
        symmetric: bool = True, # Full tables: average the two directions of each pair (the channel is reciprocal; the tracing samples each direction apart)
    ):
        z = np.load(path)
        self.cells = [tuple(int(v) for v in c) for c in z['cells']]
        self.idx = {c: i for i, c in enumerate(self.cells)}
        self.walls = z['walls']
        self.source = json.loads(str(z['source'])) if 'source' in z.files else {}
        if 'pairs' in z.files:                                     # a pair cache: each pair traced one way
            self.G = None
            self.pairs = {_pair(a, b): max(abs(complex(h)) ** 2, 1e-15) for (a, b), h in zip(z['pairs'].tolist(), z['H'])}
        else:
            G = np.abs(z['H'].astype(np.complex128)) ** 2
            self.G = np.maximum((G + G.T) / 2 if symmetric else G, 1e-15)        # G[rx, tx]; no path: -150 dB
            self.pairs = None

    def gain(self, tx, rx) -> float:
        "|h|² from cell `tx` to cell `rx` (from a pair cache: KeyError if the pair was not traced)."
        if self.pairs is not None:
            return self.pairs[_pair(tx, rx)]
        return float(self.G[self.idx[tuple(rx)], self.idx[tuple(tx)]])

def table_path(layout_dir, seed) -> Path:
    "The layout's full table (`layout_<seed>.npz`) if there is one, else its pair cache (`pairs_<seed>.npz`)."
    full = Path(layout_dir) / f'layout_{seed}.npz'
    return full if full.exists() else Path(layout_dir) / f'pairs_{seed}.npz'

## Trajectories

`collect_trajectories` rolls out `episodes` episodes in the layout of each seed, each from random starts and goal
(`reset_in_layout`), and saves `trajectories_<seed>.npz`: `cells` (episodes, steps, agents, 2) and `dirs`
(episodes, steps, agents), with -1 once an agent has left the grid (FindGoal removes an agent at the goal) or the
episode has ended, and the `behaviour` of each episode. The agents follow stable-marl's `MixedPolicy`, which draws a
behaviour per episode with the probabilities in `policy`:

* `random`: uniform actions.
* `epsilon_expert`: a shortest path to the goal, with a random action with probability `epsilon`.
* `waypoint`: a shortest path to a random cell of the agent's own, then to the goal.

The movement in the data should be the movement at deployment: the predictor learns how the gain changes when the
agents move.

In [ ]:
#| export
def collect_trajectories(
    env_cfg, # The environment's config (`make_env`)
    seeds, # The layouts' reset seeds
    out_dir, # Where `trajectories_<seed>.npz` go
    episodes: int = 5, # Episodes per layout
    max_steps: int = 150, # Steps per episode, at most
    policy: dict | None = None, # `MixedPolicy`'s behaviour -> probability; None: 0.2 random, 0.5 epsilon_expert, 0.3 waypoint
    epsilon: float = 0.3, # The random actions of `epsilon_expert`
    seed: int = 0, # Seed of the starts and the actions
    overwrite: bool = False, # Collect again the layouts already in `out_dir`
) -> list[Path]:
    "Every agent's cell and heading at every step of `episodes` episodes in the layout of each seed."
    from stable_marl.envs.multigrid.expert import MixedPolicy
    weights = _plain(policy) if policy is not None else None
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for layout in seeds:
        path = out_dir / f'trajectories_{layout}.npz'
        paths.append(path)
        if path.exists() and not overwrite:
            continue
        rng = np.random.default_rng([seed, int(layout)])
        agents_policy = MixedPolicy(weights, epsilon, seed=int(rng.integers(2 ** 63)))
        env = make_env(env_cfg)
        n = len(env.unwrapped.agents)
        cells = np.full((episodes, max_steps, n, 2), -1, np.int16)
        dirs = np.full((episodes, max_steps, n), -1, np.int8)
        used = []
        for e in range(episodes):
            reset_in_layout(env, layout, rng)
            u = env.unwrapped
            for t in range(max_steps):
                for i, agent in enumerate(u.agents):
                    if not agent.state.terminated and tuple(agent.state.pos) != (-1, -1):
                        cells[e, t, i] = agent.state.pos
                        dirs[e, t, i] = int(agent.state.dir)
                actions = agents_policy.act(u)
                if t == 0:
                    used.append(agents_policy.behaviour(u))
                _, _, terminated, truncated, _ = env.step({i: int(a) for i, a in enumerate(actions)})
                if all(terminated.values()) or all(truncated.values()):
                    break
        np.savez_compressed(path, cells=cells, dirs=dirs, behaviour=np.array(used), layout=int(layout))
    return paths

## Tracing the visited pairs

Agents visit a few percent of a layout's pairs of cells, and a Sionna RT call costs about the same per pair, so
tracing only those is far cheaper than a full table (on a 15 × 15 FindGoal grid: a few minutes per layout instead of
about 45, on a CPU). A `PairCache` keeps a layout's traced pairs in `pairs_<seed>.npz`, and traces the missing ones:

* each pair once, one way (the channel is reciprocal; this also avoids the two directions' sampling mismatch);
* grouped by transmitter, the cell of each pair shared by more pairs transmitting, so that one Sionna RT call traces
  several pairs.

`trace_visited_pairs` traces the pairs of each layout's trajectories. The cache grows as needed: more episodes, or the
pairs met in an experiment (`CachedRayTracedFading`), are traced when first needed.

In [ ]:
#| export
class PairCache:
    "A layout's traced pairs of cells, kept in `path`; `trace` traces the missing ones (one way each, grouped by transmitter)."

    def __init__(
        self,
        path, # The cache file: `<dir>/pairs_<seed>.npz` (read if it exists)
        env_cfg, # The environment's config (`make_env`)
        layout_seed: int, # The layout's reset seed
        frequency: float = 2.4e9, # Carrier frequency (Hz)
        max_depth: int = 5, # Reflections per path
        scene: dict | None = None, # `SceneConfig` fields to change
        tracer=None, # tracer(tx_cell, rx_cells) -> their complex channels; None: Sionna RT (e.g. a stand-in for tests)
        **solver_kwargs, # Other options of comm-core's `trace_paths` (e.g. batch) and Sionna RT's PathSolver
    ):
        self.path, self.env_cfg, self.seed = Path(path), env_cfg, int(layout_seed)
        self.frequency, self.max_depth, self.scene, self.solver_kwargs = frequency, max_depth, scene or {}, solver_kwargs
        self.tracer, self._scene = tracer, None
        self.env = make_env(env_cfg)
        self.env.reset(seed=self.seed)
        self.cells = free_cells(self.env)
        g = self.env.unwrapped.grid
        self.walls = np.ones((g.width, g.height), np.uint8)
        for c in self.cells:
            self.walls[c] = 0
        self.H, self.unsaved = {}, 0
        if self.path.exists():
            z = np.load(self.path)
            if [tuple(c) for c in z['cells'].tolist()] != self.cells:
                raise ValueError(f"{self.path} holds another layout than the reset seed {self.seed}'s")
            self.H = {_pair(a, b): complex(h) for (a, b), h in zip(z['pairs'].tolist(), z['H'])}

    def missing(self, pairs) -> list[tuple]:
        "The pairs (of distinct cells) not traced yet."
        return sorted({p for p in (_pair(a, b) for a, b in pairs) if p[0] != p[1] and p not in self.H})

    def trace(self, pairs, progress: bool = False) -> int:
        "Trace the missing `pairs`; the number traced."
        from collections import Counter, defaultdict
        todo = self.missing(pairs)
        degree = Counter(c for p in todo for c in p)
        groups = defaultdict(list)
        for a, b in todo:
            tx, rx = (a, b) if (degree[a], a) >= (degree[b], b) else (b, a)
            groups[tx].append(rx)
        for k, (tx, rxs) in enumerate(groups.items()):
            for rx, h in zip(rxs, self._trace(tx, rxs)):
                self.H[_pair(tx, rx)] = complex(h)
            if progress:
                print(f'{k + 1} / {len(groups)} transmitters', end='\r')
        self.unsaved += len(todo)
        return len(todo)

    def _trace(self, tx, rxs):
        if self.tracer is not None:
            return self.tracer(tx, rxs)
        import contextlib, io, os, tempfile
        import sionna.rt as rt
        from comm_core import trace_paths
        from stable_marl.envs.multigrid.sionna_export import export_scene, grid_to_world
        config = _scene_config(self.scene)
        if self._scene is None:                                   # the layout's scene, built once
            self._dir = tempfile.TemporaryDirectory()
            with contextlib.redirect_stdout(io.StringIO()):
                export_scene(self.env, self._dir.name, config, frequency=self.frequency)
            self._scene = rt.load_scene(os.path.join(self._dir.name, 'scene.xml'))
            self._scene.frequency = self.frequency
        g = self.env.unwrapped.grid
        point = lambda c: grid_to_world(c[0], c[1], g.width, g.height, config, config.antenna_height)
        a = trace_paths(self._scene, [point(tx)], [point(c) for c in rxs], max_depth=self.max_depth, **self.solver_kwargs)['a']
        return a[0].sum(-1)                                       # (receivers,), summed over the paths

    def gain(self, a, b) -> float:
        "|h|² between cells `a` and `b`, traced now if it was not."
        self.trace([(a, b)])
        return max(abs(self.H[_pair(a, b)]) ** 2, 1e-15)

    def save(self):
        "Write the cache to `path`."
        pairs = sorted(self.H)
        source = dict(env=_plain(self.env_cfg), seed=self.seed, frequency=self.frequency, max_depth=self.max_depth,
                      scene=self.scene, solver=self.solver_kwargs)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(self.path, pairs=np.array(pairs, np.int16).reshape(-1, 2, 2),
                            H=np.array([self.H[p] for p in pairs], np.complex64), cells=np.array(self.cells, np.int16),
                            walls=self.walls, source=json.dumps(source, default=str))
        self.unsaved = 0

In [ ]:
#| export
def visited_pairs(trajectories) -> set[tuple]:
    "The pairs of cells two agents were in at the same step, in a `trajectories_<seed>.npz` (a path, or the loaded arrays)."
    cells = (np.load(trajectories) if isinstance(trajectories, (str, Path)) else trajectories)['cells']
    on = cells[..., 0] >= 0
    out = set()
    for i in range(cells.shape[2]):
        for j in range(i + 1, cells.shape[2]):
            both = on[..., i] & on[..., j]
            for a, b in zip(cells[..., i, :][both].tolist(), cells[..., j, :][both].tolist()):
                if a != b:
                    out.add(_pair(a, b))
    return out

def trace_visited_pairs(
    env_cfg, # The environment's config (`make_env`)
    trajectory_dir, # The `trajectories_<seed>.npz` of `collect_trajectories`
    seeds, # The layouts' reset seeds
    out_dir, # Where the `pairs_<seed>.npz` caches are (and go)
    frequency: float = 2.4e9, # Carrier frequency (Hz)
    max_depth: int = 5, # Reflections per path
    scene: dict | None = None, # `SceneConfig` fields to change
    progress: bool = True, # Print each layout's count and time
    tracer=None, # A stand-in for Sionna RT (see `PairCache`)
    **solver_kwargs, # Other options of `trace_paths` and Sionna RT's PathSolver
) -> list[Path]:
    "Trace the pairs of cells each layout's trajectories visit, into its `PairCache` (only the pairs not traced yet)."
    import time
    paths = []
    for seed in seeds:
        cache = PairCache(Path(out_dir) / f'pairs_{seed}.npz', env_cfg, seed, frequency, max_depth, scene, tracer, **solver_kwargs)
        start = time.time()
        pairs = visited_pairs(Path(trajectory_dir) / f'trajectories_{seed}.npz')
        traced = cache.trace(pairs)
        cache.save()
        paths.append(cache.path)
        if progress:
            print(f'layout {seed}: {len(pairs)} pairs visited, {traced} traced now, in {time.time() - start:.0f} s', flush=True)
    return paths

In an experiment, `CachedRayTracedFading` is the channel comm-core uses (e.g. `LinkLevelChannel(fading=...)`): the
application places the nodes on cells (`set_positions`, as for comm-core's `RayTracedFading`), and a pair not in the
cache is traced the first time it is needed (about a second on a CPU), then kept. The cache is written every
`save_every` new pairs, and by `save`.

In [ ]:
#| export
from comm_core import Fading

class CachedRayTracedFading(Fading):
    "The channel between the nodes' cells, from a layout's `PairCache`; a pair not traced yet is traced when first needed."

    def __init__(
        self,
        cache: PairCache, # The layout's cache
        save_every: int = 50, # Write the cache after this many new pairs
    ):
        self.cache, self.save_every, self.node_positions = cache, save_every, {}

    def set_positions(self, positions):
        "Where the nodes are now: {node: (x, y)}."
        self.node_positions.update({k: tuple(int(v) for v in p) for k, p in positions.items()})

    def gain_between(self, tx_position, rx_position) -> float:
        "|h|² between two cells (inf: the same cell)."
        a, b = tuple(int(v) for v in tx_position), tuple(int(v) for v in rx_position)
        return math.inf if a == b else self.cache.gain(a, b)

    def coefficient(self, transmission, link) -> complex:
        a, b = self.node_positions[transmission.sender], self.node_positions[transmission.receiver]
        if a == b:
            return math.inf
        if self.cache.trace([(a, b)]) and self.cache.unsaved >= self.save_every:
            self.cache.save()
        h = self.cache.H[_pair(a, b)]
        return h if abs(h) ** 2 >= 1e-15 else complex(math.sqrt(1e-15))   # no path: -150 dB

    def save(self):
        "Write the cache."
        self.cache.save()

## Samples

For each directed link `j → i` and step `t ≥ history` at which both agents are on the grid (and were, `history`
steps back), the transmitter knows:

* the link's last `history` gains, fed back by the receiver (most recent first);
* whether it moved this step, and whether the receiver moved a step ago (the receiver's report is a step late);
* the receiver's cell and heading a step ago: since then, it has stayed or stepped ahead, its *possible cells*;
* the layout's walls.

`features` turns these into the predictor's inputs, for one cell the receiver may be in: the log-distance and the
distance to it, the walls in between, the past gains in dB, and the two moves. For training, it is computed for
the receiver's actual cell (`X`); for deployment, for each of its possible cells (`X_possible`, with
`possible_mask`), where the predictor takes the lowest prediction.

Optional small-scale fading on top of the table: Rayleigh, per pair of agents (the same both ways), redrawn
whenever an agent moves (a cell is many wavelengths) and drifting slowly (`fading_correlation`) while both stand
still.

In [ ]:
#| export
def db(g):
    "Gains in dB."
    return 10 * np.log10(np.maximum(np.asarray(g, float), 1e-30))

def features(walls: np.ndarray, tx, rx, past_gains, tx_moved: bool, rx_moved: bool) -> list[float]:
    "The predictor's inputs for a transmitter at `tx` and a receiver at `rx`: geometry, past gains (dB, most recent first), moves."
    distance, crossed = wall_geometry(walls, tx, rx)
    return [math.log10(max(distance, 0.5)), distance, crossed, *db(past_gains), float(tx_moved), float(rx_moved)]

In [ ]:
#| export
def _fading(cells: np.ndarray, rng: np.random.Generator, rho: float) -> np.ndarray:
    "|g_t|² of one pair of agents' Rayleigh fading over an episode's steps (cells: (T, 2, 2))."
    draw = lambda: (rng.standard_normal() + 1j * rng.standard_normal()) / np.sqrt(2)
    g, out = draw(), np.ones(len(cells))
    for t in range(len(cells)):
        if t and not np.array_equal(cells[t], cells[t - 1]):
            g = draw()
        elif t:
            g = rho * g + np.sqrt(1 - rho ** 2) * draw()
        out[t] = abs(g) ** 2
    return out

def episode_fading(cells: np.ndarray, rng: np.random.Generator, rho: float = 0.95) -> dict:
    "Each pair of agents' Rayleigh fading |g_t|² over an episode (cells: (steps, agents, 2)): {(i, j): (steps,)}, i < j, the same both ways."
    n = cells.shape[1]
    return {(i, j): _fading(cells[:, [i, j]], np.random.default_rng(rng.integers(2 ** 63)), rho)
            for i in range(n) for j in range(i + 1, n)}

def link_samples(
    table: ChannelTable, # The layout's channel
    cells: np.ndarray, # One episode's cells: (steps, agents, 2), -1 when an agent is not on the grid
    dirs: np.ndarray, # Its headings: (steps, agents)
    history: int = 4, # Past gains the transmitter knows
    fading: str | None = None, # None, or 'rayleigh' on top of the table
    fading_correlation: float = 0.95, # Of the fading between steps while both agents stand still
    rng: np.random.Generator | None = None, # The fading's randomness
) -> dict:
    "Per directed link and step: the features (to the receiver's actual cell, and to each of its possible cells) and the gain."
    rng = rng if rng is not None else np.random.default_rng(0)
    T, n = cells.shape[:2]
    fades = episode_fading(cells, rng, fading_correlation) if fading == 'rayleigh' else None
    on = (cells[..., 0] >= 0)
    X, Xp, mask, gain, link = [], [], [], [], []
    for j in range(n):
        for i in range(n):
            if i == j:
                continue
            both = on[:, i] & on[:, j]
            g = np.zeros(T)
            for t in np.flatnonzero(both):
                g[t] = table.gain(cells[t, j], cells[t, i])
            if fades is not None:
                g = g * fades[min(i, j), max(i, j)]
            for t in range(max(history, 2), T):
                if not both[t - history:t + 1].all() or not on[t - 2, i]:
                    continue
                tx, rx, rx_before = tuple(cells[t, j]), tuple(cells[t, i]), tuple(cells[t - 1, i])
                past = g[t - history:t][::-1]
                moved = (tx != tuple(cells[t - 1, j]), rx_before != tuple(cells[t - 2, i]))
                d = DIRS[int(dirs[t - 1, i])]
                ahead = (rx_before[0] + d[0], rx_before[1] + d[1])
                possible = [c for c in (rx_before, ahead) if c in table.idx and c != tx] or [rx_before]
                X.append(features(table.walls, tx, rx, past, *moved))
                rows = [features(table.walls, tx, c, past, *moved) for c in possible]
                Xp.append(rows + [rows[0]] * (2 - len(rows)))
                mask.append([True] * len(rows) + [False] * (2 - len(rows)))
                gain.append(g[t])
                link.append((j, i, t))
    F = 3 + history + 2
    return dict(X=np.array(X, float).reshape(-1, F), X_possible=np.array(Xp, float).reshape(-1, 2, F),
                possible_mask=np.array(mask, bool).reshape(-1, 2), gain=np.array(gain, float), link=np.array(link, int).reshape(-1, 3))

In [ ]:
#| export
def build_dataset(
    layout_dir, # The layouts' channels: `layout_<seed>.npz` (`trace_layouts`) or `pairs_<seed>.npz` (`trace_visited_pairs`)
    trajectory_dir, # The `trajectories_<seed>.npz` of `collect_trajectories`
    seeds, # The layouts to use
    history: int = 4, # Past gains the transmitter knows
    fading: str | None = None, # None, or 'rayleigh'
    fading_correlation: float = 0.95, # Of the fading while both agents stand still
    symmetric: bool = True, # Average the two directions of each traced pair
    seed: int = 0, # Seed of the fading
) -> dict:
    "`link_samples` of every episode in the layouts of `seeds`, concatenated, with each sample's `layout`."
    parts = []
    for layout in seeds:
        table = ChannelTable(table_path(layout_dir, layout), symmetric=symmetric)
        z = np.load(Path(trajectory_dir) / f'trajectories_{layout}.npz')
        rng = np.random.default_rng([seed, int(layout)])
        for cells, dirs in zip(z['cells'], z['dirs']):
            s = link_samples(table, cells, dirs, history, fading, fading_correlation, rng)
            s['layout'] = np.full(len(s['gain']), int(layout))
            parts.append(s)
    return {k: np.concatenate([p[k] for p in parts]) for k in parts[0]}

## Example

A small FindGoal (9 × 9, FindGoal's 10 single-cell walls): trajectories in two layouts, then their visited pairs
traced with a stand-in for Sionna RT (free-space path loss and 6 dB per wall crossed), so that the example runs
without it. With Sionna RT, leave out `tracer`.

In [ ]:
import tempfile

In [ ]:
env_cfg = {'_target_': 'stable_marl.envs.multigrid.FindGoalEnv', 'agents': 2, 'size': 9, 'num_obstacles': 0, 'max_steps': 60}
tmp = Path(tempfile.mkdtemp())

def stand_in_table(seed, path):
    "Instead of tracing: free-space loss at 2.4 GHz (2 m cells) and 6 dB per wall crossed."
    env = make_env(env_cfg)
    env.reset(seed=seed)
    cells, g = free_cells(env), env.unwrapped.grid
    walls = np.array([[0 if (x, y) in set(cells) else 1 for y in range(g.height)] for x in range(g.width)], np.uint8)
    H = np.zeros((len(cells), len(cells)), complex)
    for r, b in enumerate(cells):
        for s, a in enumerate(cells):
            if a != b:
                d, w = wall_geometry(walls, a, b)
                H[r, s] = 0.125 / (4 * np.pi * 2 * d) * 10 ** (-6 * w / 20)
    np.savez_compressed(path, H=H, cells=np.array(cells), walls=walls, source=json.dumps({'seed': seed}))

def stand_in(walls):
    "A tracer for `PairCache`: free-space loss at 2.4 GHz (2 m cells) and 6 dB per wall crossed."
    def tracer(tx, rxs):
        return [0.125 / (4 * np.pi * 2 * d) * 10 ** (-6 * w / 20) for d, w in (wall_geometry(walls, tx, rx) for rx in rxs)]
    return tracer

collect_trajectories(env_cfg, [0, 1], tmp, episodes=3, max_steps=40, policy={'random': 0.5, 'epsilon_expert': 0.5})
for seed in (0, 1):
    walls = PairCache(tmp / 'unused.npz', env_cfg, seed).walls
    trace_visited_pairs(env_cfg, tmp, [seed], tmp, tracer=stand_in(walls))
data = build_dataset(tmp, tmp, [0, 1], fading='rayleigh')
print({k: v.shape for k, v in data.items()})

In [ ]:
table, z = ChannelTable(tmp / 'pairs_0.npz'), np.load(tmp / 'trajectories_0.npz')
assert z['cells'].shape == (3, 40, 2, 2) and set(z['behaviour']) <= {'random', 'epsilon_expert'}
on = z['cells'][..., 0] >= 0
assert all(tuple(c) in table.idx for c in z['cells'][on])                      # agents only on free cells
assert set(table.pairs) == visited_pairs(z)                                     # exactly the visited pairs
assert len({tuple(map(tuple, z['cells'][e, 0])) for e in range(3)}) == 3        # different starts in one layout
assert data['X'].shape[1] == 3 + 4 + 2 and (data['possible_mask'][:, 0]).all()
s = link_samples(table, z['cells'][0], z['dirs'][0])                            # no fading: the table's gains
assert np.allclose(s['gain'], [table.gain(z['cells'][0, t, j], z['cells'][0, t, i]) for j, i, t in s['link']])

The cache in an experiment: a pair never visited is traced when first needed, and kept.

In [ ]:
from comm_core import Link, Message, Transmission

In [ ]:
cache = PairCache(tmp / 'pairs_0.npz', env_cfg, 0, tracer=stand_in(table.walls))
fading = CachedRayTracedFading(cache)
new = next((a, b) for a in cache.cells for b in cache.cells if a != b and _pair(a, b) not in cache.H)
fading.set_positions({'agent_0': new[0], 'agent_1': new[1]})
transmission = Transmission(Message('agent_0', 'agent_1', payload=0), 'agent_0', 'agent_1', start_time=0)
before = len(cache.H)
h = fading.coefficient(transmission, Link('agent_0', 'agent_1'))
assert len(cache.H) == before + 1 and np.isclose(abs(h) ** 2, cache.gain(*new))
fading.coefficient(transmission, Link('agent_0', 'agent_1'))                   # now cached: nothing traced
assert len(cache.H) == before + 1 and cache.missing([new]) == []
fading.save()
assert _pair(*new) in ChannelTable(tmp / 'pairs_0.npz').pairs